# 05 - Mapping Strategy

## Purpose

- The purpose of this notebook is to define and validate the mapping strategy between the source datasets and the target Abicart import format.
- This includes identifying how fields from the Master List, Price List, Product Feed, and existing Abicart export correspond to the final import schema. The notebook documents all mapping decisions, highlights ambiguities and records assumptions that must be validated before any transformations are implemented.
- No data transformations are performed in this notebook. Its sole purpose is to establish a reproducible and well-documented mapping specification for subsequent notebooks.

## Imports

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path

import duckdb
import pandas as pd

## File Configuration

In [ ]:
SUPPLIER = "snickers"

MASTER_LIST_PATH = (
    f"../data/{SUPPLIER}/master_list/Snickers_Masterlista_2026.csv"
)

PRICE_LIST_PATH = (
    f"../data/{SUPPLIER}/price_list/Prislista_Snickers_WW_202609.xlsx"
)

PRODUCT_FEED_PATH = (
    f"../data/{SUPPLIER}/product_feeds/PP_Export_Snickers_018_sv.xml"
)

ABICART_EXPORT_PATH = (
    f"../data/{SUPPLIER}/abicart_exports/webshop_articles_64160_latin1.csv"
)

## Raw File Inspection

In [ ]:
print("Master List:", MASTER_LIST_PATH)
print("Price List:", PRICE_LIST_PATH)
print("Product Feed:", PRODUCT_FEED_PATH)
print("Abicart Export:", ABICART_EXPORT_PATH)

In [ ]:
for path in [
    MASTER_LIST_PATH,
    PRICE_LIST_PATH,
    PRODUCT_FEED_PATH,
    ABICART_EXPORT_PATH,
]:
    print(f"{Path(path).name}: {Path(path).exists()}")

## Load Datasets

In [ ]:
duckdb.sql(f"""
CREATE OR REPLACE TABLE master_list AS 
SELECT *
FROM read_csv_auto('{MASTER_LIST_PATH}')
""")

price_list = pd.read_excel(
    PRICE_LIST_PATH,
    header=1,
)

tree = ET.parse(PRODUCT_FEED_PATH)
root = tree.getroot()

abicart_df = pd.read_csv(
    ABICART_EXPORT_PATH,
    encoding="latin1",
    skiprows=1,
    header=None,
)

## Dataset Overview

In [ ]:
print("Master List")
print(duckdb.sql("SELECT * FROM master_list LIMIT 10").df())

print("\nPrice List")
print(price_list.head())

print("\nProduct Feed")
product_elements = root.findall("ProductInfo")
number_of_products = len(product_elements)
print(f"Number of products: {number_of_products}")

print("\nAbicart")
print(abicart_df.head())

## Schema Analysis

In [ ]:
print("Master List Columns")
print(duckdb.sql("DESCRIBE master_list").df())

In [ ]:
print("Price List Columns")
print(price_list.columns.tolist())

In [ ]:
print("Product Feed Structure")
print([child.tag for child in product_elements[0]])

In [ ]:
print("Abicart Columns")
print(abicart_df.columns.tolist())

## Mapping Analysis

### Product Identifier Analysis
The purpose of this section is to determine how products are identified across the source datasets and how these identifiers should be mapped to the target Abicart import structure.

In [ ]:
print("Master List")
print(
    duckdb.sql("""
    SELECT Modell, Produktnamn
    FROM master_list
    LIMIT 10
    """).df()
)

In [ ]:
print("Price List")
print(
    price_list[
        ["Artikelnr", "Modell", "Färg", "Storlekskod"]
    ].head(10)
)

In [ ]:
print("Product Feed")

for product in product_elements:
    if product.findtext("ModelCode") == "1100":
        print(
            {
                "StockCode": product.findtext("StockCode"),
                "ModelCode": product.findtext("ModelCode"),
                "ColourCode": product.findtext("ColourCode"),
                "Size": product.findtext("Size"),
            }
        )
        break

Observed:
- The master List identifies products at model level using `Modell`.
- The Price List identifies variants using `Artikelnr`.
- The Product Feed uses the same variant identifier in `StockCode`.
- For the inspected example, `StockCode` consists of `ModelCode`, `ColourCode` and `Size`.

### Abicart Product Structure

In [ ]:
print("Abicart")

print(abicart_df.head(10))

Observed:
- Abicart appears to use a parent/variant structure.
- The parent product is represented by a shared product identifier in column `0`.
- Parent rows contain product metadata such as name, description, price, image and categories.
- Child rows contain variant identifiers in column `13`.
- `StockCode` in the Product Feed uniquely identifies a product variant.
- Abicart uses a similar concept for variant identification, although represented using a parent-suffix structure (e.g, `34-874-6`).

### Product Name Analysis
The purpose of this section is to determine how product names are represented across the source datasets and how they should be mapped to the target Abicart import structure.

In [ ]:
print("Master List")

print(
    duckdb.sql("""
    SELECT Modell, Produktnamn
    FROM master_list
    LIMIT 10
    """).df()
)

In [ ]:
print("Price List")

print(
    price_list[
        ["Artikelnr", "Beskrivning 1", "Beskrivning 2"]
    ].head(10)
)

In [ ]:
print("Product Feed")

for product in product_elements:
    if product.findtext("ModelCode") == "1100":
        print(
            {
                "name": product.findtext("Name"),
                "ProdDesc1": product.findtext("ProdDesc1"),
                "ProdDesc2": product.findtext("ProdDesc2"),            
            }
        )
        break

Observed:
- The Master List uses `Produktnamn` as the model-level product name.
- The Price List uses `Beskrivning 1` as the product name and `Beskrivning 2` for variant information.
- In the Product Feed, `Name` contains a more descriptive marketing name.
- `ProdDesc1` contains a shorter product name in uppercase.
- `ProdDesc2` contains variant information such as a colour and size.

In [ ]:
print("Abicart")

print(
    abicart_df[
        [0, 4]
    ].head(10)
)

Observed:
- Abicart stores the product name in column `4` for parent products.
- Variant rows do not contain product names and instead inherit product information from the parent product.

### Description Analysis
The purpose of this section is to determine how product description are represented across the source datasets and how they should be mapped to the target Abicart import structure.

In [ ]:
print("Master List")

print(
    duckdb.sql("""
    SELECT *
    FROM master_list
    LIMIT 5
    """).df()
)

Observed:
- The Master List does not contain a dedicated product description field.
- The `Kommentar` field exists but is empty for the inspected records.

In [ ]:
print("Price List")

print(
    price_list[
        ["Beskrivning 1", "Beskrivning 2"]
        ].head(5)
)

Observed: 
- The Price List does not contain a dedicated product description field.
- `Beskrivning 1` contains the product name.
- `Beskrivning 2` contains variant information such as colour and size.

In [ ]:
print("Product Feed")

for product in product_elements:
    if product.findtext("ModelCode") == "1100":
        print(
            {
                "Intro": product.findtext("Intro"),
                "TechnicalDescription": product.findtext(
                    "TechnicalDescription"
                ),
                "Feature1": product.findtext("Feature1"),
                "Feature2": product.findtext("Feature2"),
            }
        )
        break

Observed:

- The Product Feed contains dedicated fields for product descriptions and marketing content.
- `Intro` contains a short product description.
- `TechnicalDescription` contains a more detailed product description.
- `Feature1` and subsequent feature fields contain product selling points and technical highlights.

In [ ]:
print("Abicart")

print(
    abicart_df[
        [0, 5]
    ].head(10)
)

Observed: 
- Abicart stores the product description in column `5`.
- Product description are stored at the parent level.
- Variant rows do not contain descriptions and inherit them from the parent product. 

In [ ]:
abicart_df.notnull().sum()

Additional observations:
- Non-null counts support the observed parent/variant structure in the Abicart export.
- Columns `0`, `5` and `13` appear to play a central roles in product identifications, descriptions and variant handling.

### Price Analysis
The purpose of this section is to to determine how product prices are represented across the source datasets and how they should be mapped to the target Abicart import structure.

In [ ]:
print("Master List")

print(
    duckdb.sql("""
    SELECT
        Modell,
        Produktnamn,
        "Lägsta RRP 2026",
        "Högsta RRP 2026"
    FROM master_list
    LIMIT 10
    """).df()
)

In [ ]:
duckdb.sql("""
SELECT
    COUNT(*) AS number_of_models
FROM master_list
WHERE "Lägsta RRP 2026" != "Högsta RRP 2026"
""").df()

Observed:
- The master List contains recommended retail prices (RRP) at the model level.
- `Lägsta RRP 2026` and `Högsta RRP 2026` are identical for all records in the Master List.
- No model-level price ranges were identified.

In [ ]:
print("Price List")

print(
    price_list[
        ["Artikelnr", "Nettopris", "RRP Pris"]
    ].head(10)
)

Observed:
- The Master List contains a single RRP value per model.
- The Price List contains prices at the variant level.
- Standard variants share the same RRP as the corresponding model.
- Special variants (e.g., `Reg`, `Kort`) may have different net prices and RRPs.

In [ ]:
print("Product Feed")

for product in product_elements:
    if product.findtext("ModelCode") == "1100":
        print(
            [child.tag for child in product]
        )
        break

Observed:
- No dedicated price fields were identified in the Product Feed structure.
- The Product Feed appears to contain product, variant, technical and marketing information rather than pricing data.

In [ ]:
abicart_df[[0, 6]].head(10)

Observed: 
- Abicart stores product prices in column `6`.
- Unlike product names and descriptions, prices are repeated across both parent and variant rows. 

### EAN Analysis
The purpose of this section is to determine how EAN information is represented across the source datasets and how it should be mapped to the target Abicart import structure. 

In [ ]:
print("Price List")

print(
    price_list[
        ["Artikelnr", "EAN-nr. styck", "EAN-nr. förpackning"]
    ].head(10)
)

In [ ]:
print("Product Feed")

for product in product_elements:
    if product.findtext("ModelCode") == "1100":
        print(
            {
                "StockCode": product.findtext("StockCode"),
                "EAN_text" : product.findtext("EAN_text"),
            }
        )
        break

In [ ]:
abicart_df.notnull().sum()

In [ ]:
print(abicart_df[[8, 9, 10, 11, 12, 13]].head(10))

Observed:
- Master List: No EAN field.
- Price List: EAN-nr. styck.
- Product Feed: EAN_text.
- Abicart: No EAN field identified.

### Image Analysis
The purpose of this section is to determine how product images are represented across the source datasets and how they should be mapped to the target Abicart import structure.

Observed:
- No image-related fields were identified in the Master List structure.
- No image-related fields were identified in the Price List structure.

In [ ]:
print("Product Feed")

for product in product_elements:
    if product.findtext("ModelCode") == "1100":
        print(
            {
                "MainImage": product.findtext(
                    "This_x0020_is_x0020_the_x0020_main_x0020_image"
                )
            }
        )
        break

print("\nAbicart")

print(abicart_df[[0, 8]].head(10))

Observed: 
- The Product Feed contains a dedicated main image field with an image URL:
- In the inspected Abicart export, image URLs are stored in column 8 on parent product rows.

| Target Field | Master List | Price List | Product Feed | Abicart | Notes |
|--------------|-------------|------------|--------------|---------|-------|
| Product Identifier | Modell | Artikelnr | StockCode | Parent ID / Variant ID | Model-level vs variant-level identifiers |
| Product Name | Produktnamn | Beskrivning 1 | Name | Column 4 | Product names represented differently across systems |
| Description | - | - | Intro | Column 5 | Description stored at parent level |
| Price | Lägsta/Högsta RRP 2026 | Nettopris, RRP Pris | - | Column 6 | Model-level, variant-level, and webshop prices |
| Image | - | - | Main image field | Column 8 | Product Feed provides the source image URL |

## Data Quality Checks

- No duplicate `StockCode` values were identified in the inspected Product Feed records.
- No duplicate `Artikelnr` values were identified in the inspected Price List records.
- `EAN_text` and `EAN-nr. styck` were consistent for the inspected records.
- `Lägsta RRP 2026` and `Högsta RRP 2026` were identical for the inspected models in the Master List.
- Special variants (`Reg`, `Kort`) with different pricing were identified in the Price List and require filtering during transformation.
- No image-related fields were identified in the Master List or Price List during schema analysis.

## Findings

- Product identification exists at both model and variant level across the source datasets.
- The Price List serves as the primary source for variant-level pricing and EAN information.
- The Product Feed serves as the primary source for descriptions, images and marketing content.
- The Abicart export uses a parent/variant structure.
- EAN information is available from the supplier but is not present in the exported Abicart dataset.

## Open Questions

- Should EAN values be included in future webshop imports?
- Are additional Product Feed fields (e.g., sustainability and certification fields) relevant for webshop presentation?
- Should future versions of the importer support special variants (`Reg`, `Kort`) that are currently excluded?
- Is additional category mapping required to support webshop navigation?